In [133]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
#from utils import *
#from utils_cpdir import *
from utils_mvcp import *
from utils import *
from utils_spec import *
import os
from tqdm import tqdm
import json

In [134]:
remove_slopp = pd.read_excel('RochmanSpectra_ToRemove.xlsx', sheet_name="SLOPPe Remove")

mIRageLibraySpectra_Removed = pd.read_csv('../gloves_contamination/mIRageLibraySpectra_Removed.csv')
mIRageLibraySpectra_Removed.iloc[-1,2] = 'PE_15'

dict_slopp_reb = {'PE':'Polyethylene', 
                  'PP':'Polypropylene', 
                  'PSL':'Polystyrene'}

with open('passive_data_raman_saturation.json', 'r') as f:
    saturated_wn_dict = json.load(f)

    
ids_sat = list(saturated_wn_dict.keys())

slopp_df = pd.read_csv('../enviromental_data/slopp_data_raman_final_specy.csv')
del slopp_df['Unnamed: 0']
slopp_df.columns = list(np.float64(slopp_df.columns[:-4])) + list(slopp_df.columns[-4:])

slopp_df = slopp_df.query('polymer != "Cotton" and polymer !="Dyed Cellulose"').reset_index(drop=True)
slopp_df.loc[slopp_df['polymer'] == "Polyester",'polymer'] = "Polyethylene Terephthalate"
slopp_df = slopp_df.loc[[x not in np.array(remove_slopp['Example']) for x in slopp_df['id']], :].reset_index(drop=True)

rebecca_raman_df = pd.read_csv('../enviromental_data/rebecca_data_raman_final_specy.csv')
del rebecca_raman_df['Unnamed: 0']
rebecca_raman_df.columns = list(np.float64(rebecca_raman_df.columns[:-4])) + list(rebecca_raman_df.columns[-4:])

rebecca_raman_df = rebecca_raman_df.drop(np.where([x in list(mIRageLibraySpectra_Removed.query('Signal == "Raman"')['ID']) for x in rebecca_raman_df['id']])[0],
                             axis=0).reset_index(drop=True)

rebecca_raman_df = rebecca_raman_df.query('polymer in @dict_slopp_reb').reset_index(drop=True)
rebecca_raman_df['polymer'] = [dict_slopp_reb[x] for x in rebecca_raman_df['polymer']]

passive_data_raman = pd.read_csv('passive_data_raman_specy.csv')
del passive_data_raman['Unnamed: 0']
del passive_data_raman['wn_saturated']
del passive_data_raman['id_file']
ids_no_signal = np.array(passive_data_raman.query('no_signal>0')['id'])
del passive_data_raman['no_signal']
passive_data_raman.columns = list(np.float64(passive_data_raman.columns[:-3])) + list(passive_data_raman.columns[-3:])
passive_data_raman['polymer'] = ''
passive_data_raman['label'] = ''

slopp_df.iloc[:,:-4] = (slopp_df.iloc[:,:-4] - slopp_df.iloc[:,:-4].mean(1)[:,None]) / slopp_df.iloc[:,:-4].std(1)[:,None]
rebecca_raman_df.iloc[:,:-4] = (rebecca_raman_df.iloc[:,:-4] - rebecca_raman_df.iloc[:,:-4].mean(1)[:,None]) / rebecca_raman_df.iloc[:,:-4].std(1)[:,None]
passive_data_raman.iloc[:,:-5] = (passive_data_raman.iloc[:,:-5] - passive_data_raman.iloc[:,:-5].mean(1)[:,None]) / passive_data_raman.iloc[:,:-5].std(1)[:,None]

idx_raman_nan = np.unique(np.where(np.isnan(passive_data_raman.iloc[:,:-5]))[0])

In [135]:
remove_flopp = pd.read_excel('RochmanSpectra_ToRemove.xlsx', sheet_name="FLOPPe Remove")

flopp_df = pd.read_csv('../enviromental_data/flopp_data_ir_final.csv')
del flopp_df['Unnamed: 0']
flopp_df.columns = list(np.float64(flopp_df.columns[:-4])) + list(flopp_df.columns[-4:])
flopp_df = flopp_df.query('polymer != "Cotton"').reset_index(drop=True)
flopp_df.columns = list(np.float64(flopp_df.columns[:-4])) + list(flopp_df.columns[-4:])
flopp_df = flopp_df.loc[[x not in np.array(remove_flopp['Example']) for x in flopp_df['id']], :].reset_index(drop=True)

rebecca_ir_df = pd.read_csv('../enviromental_data/rebecca_data_ir_final.csv')
del rebecca_ir_df['Unnamed: 0']
rebecca_ir_df.columns = list(np.float64(rebecca_ir_df.columns[:-4])) + list(rebecca_ir_df.columns[-4:])

rebecca_ir_df = rebecca_ir_df.drop(np.where([x in list(mIRageLibraySpectra_Removed.query('Signal == "PTIR"')['ID']) for x in rebecca_ir_df['id']])[0],
                             axis=0).reset_index(drop=True)

rebecca_ir_df = rebecca_ir_df.query('polymer in @dict_slopp_reb').reset_index(drop=True)
rebecca_ir_df['polymer'] = [dict_slopp_reb[x] for x in rebecca_ir_df['polymer']]


passive_data_ir = pd.read_csv('passive_data_ir.csv')
del passive_data_ir['Unnamed: 0']
background_ids = np.array(passive_data_ir.query('background == "background"')['id'])
passive_data_ir.columns = list(np.float64(passive_data_ir.columns[:-4])) + list(passive_data_ir.columns[-4:])
passive_data_ir['polymer'] = ''
passive_data_ir['label'] = ''

flopp_df.iloc[:,:-4] = (flopp_df.iloc[:,:-4] - flopp_df.iloc[:,:-4].mean(1)[:,None]) / flopp_df.iloc[:,:-4].std(1)[:,None]
rebecca_ir_df.iloc[:,:-4] = (rebecca_ir_df.iloc[:,:-4] - rebecca_ir_df.iloc[:,:-4].mean(1)[:,None]) / rebecca_ir_df.iloc[:,:-4].std(1)[:,None]
passive_data_ir.iloc[:,:-6] = (passive_data_ir.iloc[:,:-6] - passive_data_ir.iloc[:,:-6].mean(1)[:,None]) / passive_data_ir.iloc[:,:-6].std(1)[:,None]

In [136]:
passive_data_ir_paired = passive_data_ir.query('id not in @ids_sat and id not in @background_ids and id not in @ids_no_signal').reset_index(drop=True)
passive_data_ir_unpaired_ram = passive_data_ir.query('(id not in @ids_sat and id not in @ids_no_signal) and id in @background_ids').reset_index(drop=True)
passive_data_ir_unpaired_ir = passive_data_ir.query('(id in @ids_sat or id in @ids_no_signal) and id not in @background_ids').reset_index(drop=True)
passive_data_ir_nopair = passive_data_ir.query('(id in @ids_sat or id in @ids_no_signal) and id in @background_ids').reset_index(drop=True)

passive_data_raman_paired = passive_data_raman.query('(id not in @ids_sat and id not in @ids_no_signal) and id not in @background_ids').reset_index(drop=True)
passive_data_raman_unpaired_ram = passive_data_raman.query('(id not in @ids_sat and id not in @ids_no_signal) and id in @background_ids').reset_index(drop=True)
passive_data_raman_unpaired_ir = passive_data_raman.query('(id in @ids_sat or id in @ids_no_signal) and id not in @background_ids').reset_index(drop=True)
passive_data_raman_nopair = passive_data_raman.query('(id in @ids_sat or id in @ids_no_signal) and id in @background_ids').reset_index(drop=True)


In [137]:
print('unpaired Raman spectra: ', passive_data_ir_unpaired_ram.shape[0])
print('unpaired IR spectra: ', passive_data_ir_unpaired_ir.shape[0])
print('no pair spectra: ', passive_data_ir_nopair.shape[0])
print('paired spectra: ', passive_data_ir_paired.shape[0])

unpaired Raman spectra:  1824
unpaired IR spectra:  2096
no pair spectra:  193
paired spectra:  4073


In [138]:
print('raman spectra with no signal: ', ids_no_signal.shape[0]),
print('raman spectra with saturation: ', len(ids_sat))
print('ir spectra with background: ', len([x for x in background_ids if x not in list(ids_no_signal) + ids_sat]))
print('rest spectra: ', passive_data_raman_paired.shape[0])

raman spectra with no signal:  504
raman spectra with saturation:  1785
ir spectra with background:  1824
rest spectra:  4073


In [139]:
slopp_df.groupby('polymer').count().iloc[:,-1:].merge(flopp_df.groupby('polymer').count().iloc[:,-1:], 
                                                      left_index=True,
                                                      right_index=True, how='outer')

,id_x,id_y
polymer,,
Acrylic,8.0,5.0
Acrylonitrile Butadiene Styrene,10.0,14.0
Cellulose Acetate,3.0,2.0
Ethylene Acrylic Acid,NaN,1.0
Polyamide,13.0,18.0
Polybutylene Terephthalate,1.0,NaN
Polycarbonate,8.0,8.0
Polyethylene,43.0,65.0
Polyethylene Adipate,NaN,1.0


In [140]:
flopp_slopp_labels = np.array(slopp_df.groupby('polymer').count().iloc[:,-1:].merge(flopp_df.groupby('polymer').count().iloc[:,-1:], 
                                                      left_index=True,
                                                      right_index=True, how='inner').index)

flopp_df = flopp_df.query('polymer in @flopp_slopp_labels').reset_index(drop=True)
slopp_df = slopp_df.query('polymer in @flopp_slopp_labels').reset_index(drop=True)


In [141]:
slopp_df.groupby('polymer').count().iloc[:,-1:].merge(flopp_df.groupby('polymer').count().iloc[:,-1:], 
                                                      left_index=True,
                                                      right_index=True, how='outer')

,id_x,id_y
polymer,,
Acrylic,8,5
Acrylonitrile Butadiene Styrene,10,14
Cellulose Acetate,3,2
Polyamide,13,18
Polycarbonate,8,8
Polyethylene,43,65
Polyethylene Terephthalate,32,47
Polyethylene Vinyl Acetate,2,13
Polymethyl Methacrylate,3,2


In [142]:
idx_both_rebecca = np.array(rebecca_ir_df.loc[[x in np.array(rebecca_raman_df['id']) for x in rebecca_ir_df['id']],'id'])# 
rebecca_ir_df = rebecca_ir_df.query('id in @idx_both_rebecca').reset_index(drop=True)
rebecca_raman_df = rebecca_raman_df.query('id in @idx_both_rebecca').reset_index(drop=True)


# Paired

In [143]:
(reference_library_raman, 
 calibration_set_raman, 
 test_set_raman, 
 label_table_raman) = gen_data_sets(slopp_df, 
                              rebecca_raman_df, 
                              passive_data_raman_paired)

(X_train_raman, X_cal_raman, X_test_raman, 
 y_train_raman, y_cal_raman, y_test_raman, 
 label_train_raman, label_cal_raman, label_test_raman, 
 n_test_raman, wl_list_raman) = process_data(reference_library_raman,
                                             calibration_set_raman,
                                             test_set_raman)




In [144]:
(reference_library_ir, 
 calibration_set_ir, 
 test_set_ir, 
 label_table_ir) = gen_data_sets_ir(flopp_df, 
                              rebecca_ir_df, 
                              passive_data_ir_paired)

(X_train_ir, X_cal_ir, X_test_ir, 
 y_train_ir, y_cal_ir, y_test_ir, 
 label_train_ir, label_cal_ir, label_test_ir, 
 n_test_ir, wl_list_ir) = process_data_ir(reference_library_ir,
                                 calibration_set_ir,
                                 test_set_ir)

In [145]:
pred_slopp_df = corr_matching(np.array(slopp_df.iloc[:,:-4]), 
                              slopp_df['polymer'], 
                              np.array(calibration_set_raman.iloc[:,3:]), r_min=0)

pred_flopp_df = corr_matching(np.array(flopp_df.iloc[:,:-4]), 
                              flopp_df['polymer'], 
                              np.array(calibration_set_ir.iloc[:,3:]), r_min=0)

(np.mean(np.array(pred_slopp_df[0]) == np.array(calibration_set_raman['polymer'])),
 np.mean(np.array(pred_flopp_df[0]) == np.array(calibration_set_ir['polymer'])))

(0.4878048780487805, 0.6292682926829268)

In [146]:
pred_slopp_df = corr_matching(np.array(slopp_df.iloc[:,:-4]), 
                              slopp_df['polymer'], 
                              np.array(test_set_raman.iloc[:,3:]), r_min=0)

pred_flopp_df = corr_matching(np.array(flopp_df.iloc[:,:-4]), 
                              flopp_df['polymer'], 
                              np.array(test_set_ir.iloc[:,3:]), r_min=0)

In [147]:
#pred_flopp_df = matching(flopp_df, passive_data_ir.iloc[:,:-2], limits=((980,1800),(2720, 2970)), lib_name='ref', data_name='final')
#pred_slopp_df = matching_r(slopp_df, test_set_raman.iloc[:,list(range(3,test_set_raman.shape[1]))+[0]], limits=(600,3500), lib_name='ref', data_name='final')


In [148]:
alpha = 0.1

In [149]:
# Calculate HQI for calibration set
smx_cal = smx_nn(X_train_ir, X_cal_ir, label_train_ir)

# Calculate HQI for test set
smx_test = smx_nn(X_train_ir, X_test_ir, label_train_ir)

# Set confidence level 1 - alpha
n = smx_cal.shape[0]
#alpha = 0.1
#for alpha in [alpha]:
prediction_sets, qhat = conform_pred(smx_cal, smx_test, label_cal_ir, n, alpha, cond=0)

prediction_sets_n = [list(label_table_ir.loc[ps,'polymer']) for ps in prediction_sets]
prediction_sets_size = np.array([len(ps) for ps in prediction_sets_n])

zero_label = np.where(prediction_sets_size == 0)[0]
single_label = np.where(prediction_sets_size == 1)[0]
mult_label = np.where(prediction_sets_size > 1)[0]

print('Sets with zero labels: ' + str(zero_label.shape[0]))
print('Sets with one label: ' + str(single_label.shape[0]))
print('Sets with >1 labels: ' + str(mult_label.shape[0]))
print('Mean set size:', prediction_sets.sum(1).mean())
print('Std set size:', prediction_sets.sum(1).std())
print('HQI cutt off:', 1-qhat)

Sets with zero labels: 0
Sets with one label: 547
Sets with >1 labels: 3526
Mean set size: 4.219985268843605
Std set size: 3.485823881291383
HQI cutt off: 0.8938658598510795


In [150]:
# Calculate HQI for calibration set
smx_cal = smx_nn(X_train_raman, 
                  X_cal_raman, 
                  label_train_raman)

# Calculate HQI for test set
smx_test = smx_nn(X_train_raman, 
                   X_test_raman, 
                   label_train_raman)

# Set confidence level 1 - alpha
n = smx_cal.shape[0]
#for alpha in [alpha]:
prediction_sets, qhat = conform_pred(smx_cal, smx_test, label_cal_raman, n, alpha, cond=0)

prediction_sets_n_raman = [list(label_table_raman.loc[ps,'polymer']) for ps in prediction_sets]
prediction_sets_size = np.array([len(ps) for ps in prediction_sets_n_raman])

zero_label = np.where(prediction_sets_size == 0)[0]
single_label = np.where(prediction_sets_size == 1)[0]
mult_label = np.where(prediction_sets_size > 1)[0]

print('Sets with zero labels: ' + str(zero_label.shape[0]))
print('Sets with one label: ' + str(single_label.shape[0]))
print('Sets with >1 labels: ' + str(mult_label.shape[0]))
print('Mean set size:', prediction_sets.sum(1).mean())
print('Std set size:', prediction_sets.sum(1).std())
print('HQI cutt off:', 1-qhat)

Sets with zero labels: 0
Sets with one label: 20
Sets with >1 labels: 4053
Mean set size: 13.776577461330714
Std set size: 1.4182602679093526
HQI cutt off: 0.8649577406258107


In [151]:
smx_all_IR = 1-np.power(smx_nn(X_train_ir, X_cal_ir, label_train_ir),1)
smx_all_Raman = 1-np.power(smx_nn(X_train_raman, X_cal_raman, label_train_raman),1)

smx_all_IR_test = 1-np.power(smx_nn(X_train_ir, X_test_ir, label_train_ir),1)
smx_all_Raman_test = 1-np.power(smx_nn(X_train_raman, X_test_raman, label_train_raman),1)


score_list = [smx_all_IR, smx_all_Raman]
score_list_test = [smx_all_IR_test, smx_all_Raman_test]

#alpha = .1
seed = 1
K = 2
quant_smx_list = []
rest_smx_list = []
cal_smx_list = []
val_smx_list = []
U_list = gen_u(K, seed=0)

for i in range(K):
    #quant_smx, rest_smx, quant_labels, rest_labels, idx_quant = split_cal_test(score_list[i], label_ir, cal_prop=0.2, seed=seed)
    quant_smx, cal_smx, quant_labels, cal_labels, idx_quant = split_cal_test(score_list[i], label_cal_ir, cal_prop=0.1, seed=seed)
    
    val_smx, val_labels = score_list_test[i], None
    
    #y_val = list(label_table.iloc[val_labels,1])
    
    #y_val_ir = np.array(notes_mix_std.loc[~idx_quant,:].loc[~idx,'polymer_ir'])
    #y_val_raman = np.array(notes_mix_std.loc[~idx_quant,:].loc[~idx,'polymer_raman'])
    
    quant_smx_list.append(quant_smx)
    #rest_smx_list.append(rest_smx)
    cal_smx_list.append(cal_smx)
    val_smx_list.append(val_smx)

n = cal_labels.shape[0]

prediction_sets = mvcp_2(quant_smx_list, 
                                             quant_labels,   
                                             cal_smx_list,   
                                             cal_labels,   
                                             val_smx_list,   
                                             alpha,     
                                             val_labels,                 
                                             U=U_list[0])

prediction_sets_n_mvcp = [list(label_table_raman.loc[ps,'polymer']) for ps in prediction_sets]
prediction_sets_size = np.array([len(ps) for ps in prediction_sets_n_mvcp])

zero_label = np.where(prediction_sets_size == 0)[0]
single_label = np.where(prediction_sets_size == 1)[0]
mult_label = np.where(prediction_sets_size > 1)[0]

print('Sets with zero labels: ' + str(zero_label.shape[0]))
print('Sets with one label: ' + str(single_label.shape[0]))
print('Sets with >1 labels: ' + str(mult_label.shape[0]))
print('Mean set size:', prediction_sets.sum(1).mean())
print('Std set size:', prediction_sets.sum(1).std())

Sets with zero labels: 8
Sets with one label: 100
Sets with >1 labels: 3965
Mean set size: 7.2919224159096485
Std set size: 4.4963335650812395


In [160]:
output = [prediction_sets_n[i][0] if i in single_label else 
              '--outlier--' if i in zero_label else 
              '--manual check!--' for i in range(label_test_ir.shape[0])]
output_df = pd.DataFrame({'id':test_set_ir['id'],
                          'prection_set_ir':prediction_sets_n,
                          'prection_set_raman':prediction_sets_n_raman,
                          'prection_set_mvcp':prediction_sets_n_mvcp,
                              #'output':output
                              })

output_df['polymer_flopp'] = np.array(pred_flopp_df[0])#['polymer_ref']
output_df['hqi_flopp'] = pred_flopp_df[1]#['hqi_ref']
output_df['id_top_match_flopp'] = np.array(flopp_df.iloc[np.array(pred_flopp_df[0].index),-1])

output_df['polymer_slopp'] = np.array(pred_slopp_df[0])#['polymer_ref']
output_df['hqi_slopp'] = pred_slopp_df[1]#['hqi_ref']
output_df['id_top_match_slopp'] = np.array(slopp_df.iloc[np.array(pred_slopp_df[0].index),-1])


output_df.to_csv('CP_restuls_passive_paired.csv')

In [14]:
#output_df_p = pd.read_csv('CP_restuls.csv')
#del output_df_p['Unnamed: 0']
#output_df_p

#output_df_compare = pd.merge(output_df.iloc[:,[0,4,5,6,7,8,9]], output_df_p.iloc[:,[0,4,5,6,7,8,9]],on='id')
#output_df_compare

#(np.mean([len(x) for x in output_df['prection_set_ir']]), 
#np.mean([len(x) for x in output_df['prection_set_raman']]),
#np.mean([len(x) for x in output_df['prection_set_mvcp']]))

#(np.mean([x.count(',')+1 for x in output_df_p['prection_set_ir']]), 
#np.mean([x.count(',')+1 for x in output_df_p['prection_set_raman']]),
#np.mean([x.count(',')+1 for x in output_df_p['prection_set_mvcp']]))

# Unpaired Raman

In [161]:
(reference_library_raman, 
 calibration_set_raman, 
 test_set_raman, 
 label_table_raman) = gen_data_sets(slopp_df, 
                              rebecca_raman_df, 
                              passive_data_raman_unpaired_ram)

(X_train_raman, X_cal_raman, X_test_raman, 
 y_train_raman, y_cal_raman, y_test_raman, 
 label_train_raman, label_cal_raman, label_test_raman, 
 n_test_raman, wl_list_raman) = process_data(reference_library_raman,
                                 calibration_set_raman,
                                 test_set_raman)


(reference_library_ir, 
 calibration_set_ir, 
 test_set_ir, 
 label_table_ir) = gen_data_sets_ir(flopp_df, 
                              rebecca_ir_df, 
                              passive_data_ir_unpaired_ram)

(X_train_ir, X_cal_ir, X_test_ir, 
 y_train_ir, y_cal_ir, y_test_ir, 
 label_train_ir, label_cal_ir, label_test_ir, 
 n_test_ir, wl_list_ir) = process_data_ir(reference_library_ir,
                                 calibration_set_ir,
                                 test_set_ir)



        
pred_slopp_df = corr_matching(np.array(slopp_df.iloc[:,:-4]), 
                              slopp_df['polymer'], 
                              np.array(test_set_raman.iloc[:,3:]), r_min=0)

pred_flopp_df = corr_matching(np.array(flopp_df.iloc[:,:-4]), 
                              flopp_df['polymer'], 
                              np.array(test_set_ir.iloc[:,3:]), r_min=0)


In [162]:
# Calculate HQI for calibration set
smx_cal = smx_nn(X_train_raman, 
                 X_cal_raman, 
                 label_train_raman)

# Calculate HQI for test set
smx_test = smx_nn(X_train_raman, 
                  X_test_raman, 
                  label_train_raman)

# Set confidence level 1 - alpha
n = smx_cal.shape[0]
alpha = 0.1
#for alpha in [alpha]:
prediction_sets, qhat = conform_pred(smx_cal, smx_test, label_cal_raman, n, alpha, cond=0)

prediction_sets_n_raman = [list(label_table_raman.loc[ps,'polymer']) for ps in prediction_sets]
prediction_sets_size = np.array([len(ps) for ps in prediction_sets_n_raman])

zero_label = np.where(prediction_sets_size == 0)[0]
single_label = np.where(prediction_sets_size == 1)[0]
mult_label = np.where(prediction_sets_size > 1)[0]

print('Sets with zero labels: ' + str(zero_label.shape[0]))
print('Sets with one label: ' + str(single_label.shape[0]))
print('Sets with >1 labels: ' + str(mult_label.shape[0]))
print('Mean set size:', prediction_sets.sum(1).mean())
print('Std set size:', prediction_sets.sum(1).std())

Sets with zero labels: 0
Sets with one label: 0
Sets with >1 labels: 1824
Mean set size: 13.953399122807017
Std set size: 0.5682763477109104


In [163]:
output = [prediction_sets_n[i][0] if i in single_label else 
              '--outlier--' if i in zero_label else 
              '--manual check!--' for i in range(label_test_ir.shape[0])]
output_df = pd.DataFrame({'id':test_set_ir['id'],
                          #'prection_set_ir':prediction_sets_n,
                          'prection_set_raman':prediction_sets_n_raman,
                          #'prection_set_mvcp':prediction_sets_n_mvcp,
                              #'output':output
                              })

output_df['polymer_flopp'] = np.array(pred_flopp_df[0])#['polymer_ref']
output_df['hqi_flopp'] = pred_flopp_df[1]#['hqi_ref']
output_df['id_top_match_flopp'] = np.array(flopp_df.iloc[np.array(pred_flopp_df[0].index),-1])

output_df['polymer_slopp'] = np.array(pred_slopp_df[0])#['polymer_ref']
output_df['hqi_slopp'] = pred_slopp_df[1]#['hqi_ref']
output_df['id_top_match_slopp'] = np.array(slopp_df.iloc[np.array(pred_slopp_df[0].index),-1])


output_df#.to_csv('CP_restuls_passive_paired.csv')

,id,prection_set_raman,polymer_flopp,hqi_flopp,id_top_match_flopp,polymer_slopp,hqi_slopp,id_top_match_slopp
0,FHS_passive_July2024_D6_20240813_1,"[Acrylic, Acrylonitrile Butadiene Styrene, Cel...",Polyethylene,0.144128,PE 11. Green Fragment.CSV,Polymethyl Methacrylate,0.389865,Polymethyl Methacrylate 1. Clear Sphere.txt
1,FHS_passive_July2024_D6_20240813_2,"[Acrylic, Acrylonitrile Butadiene Styrene, Cel...",Polyethylene Vinyl Acetate,0.184068,PEVA 10. Yellow Fiber.CSV,Polymethyl Methacrylate,0.376699,Polymethyl Methacrylate 1. Clear Sphere.txt
2,FHS_passive_July2024_D6_20240813_4,"[Acrylic, Acrylonitrile Butadiene Styrene, Cel...",Polypropylene,0.221259,PP 43. Green Fragment.CSV,Polymethyl Methacrylate,0.377495,Polymethyl Methacrylate 1. Clear Sphere.txt
3,FHS_passive_July2024_D6_20240813_5,"[Acrylic, Acrylonitrile Butadiene Styrene, Cel...",Polycarbonate,0.114916,PC 3. Lab Filter Film.CSV,Polymethyl Methacrylate,0.390095,Polymethyl Methacrylate 1. Clear Sphere.txt
4,FHS_passive_July2024_D6_20240813_6,"[Acrylic, Acrylonitrile Butadiene Styrene, Cel...",Polystyrene,0.201920,PS 6. White Foam.CSV,Polymethyl Methacrylate,0.375555,Polymethyl Methacrylate 1. Clear Sphere.txt
...,...,...,...,...,...,...,...,...
1819,DC_passive_MarApr2024_C9_20240612_192,"[Acrylic, Acrylonitrile Butadiene Styrene, Cel...",Polypropylene,0.339809,PP 43. Green Fragment.CSV,Polystyrene,0.413600,Polystyrene 4. White Film.txt
1820,DC_passive_MarApr2024_C9_20240612_195,"[Acrylic, Acrylonitrile Butadiene Styrene, Cel...",Polyethylene Vinyl Acetate,0.090600,PEVA 2. Orange Flip Flop Foam.CSV,Polymethyl Methacrylate,0.389106,Polymethyl Methacrylate 1. Clear Sphere.txt
1821,DC_passive_MarApr2024_C9_20240612_197,"[Acrylic, Acrylonitrile Butadiene Styrene, Cel...",Polyethylene Vinyl Acetate,0.094390,EVA 3. Grey Foam.CSV,Polymethyl Methacrylate,0.384389,Polymethyl Methacrylate 1. Clear Sphere.txt
1822,DC_passive_MarApr2024_C9_20240612_198,"[Acrylic, Acrylonitrile Butadiene Styrene, Cel...",Polyurethane,0.191978,PU 1. Orange Foam.CSV,Polystyrene,0.390152,Polystyrene 4. White Film.txt


In [ ]:
.to_csv('CP_restuls_passive_paired.csv')

# Unpaired IR

In [31]:
(reference_library_raman, 
 calibration_set_raman, 
 test_set_raman, 
 label_table_raman) = gen_data_sets(slopp_df, 
                              rebecca_raman_df, 
                              passive_data_raman_unpaired_ir)

(X_train_raman, X_cal_raman, X_test_raman, 
 y_train_raman, y_cal_raman, y_test_raman, 
 label_train_raman, label_cal_raman, label_test_raman, 
 n_test_raman, wl_list_raman) = process_data(reference_library_raman,
                                 calibration_set_raman,
                                 test_set_raman)


(reference_library_ir, 
 calibration_set_ir, 
 test_set_ir, 
 label_table_ir) = gen_data_sets_ir(flopp_df, 
                              rebecca_ir_df, 
                              passive_data_ir_unpaired_ir)

(X_train_ir, X_cal_ir, X_test_ir, 
 y_train_ir, y_cal_ir, y_test_ir, 
 label_train_ir, label_cal_ir, label_test_ir, 
 n_test_ir, wl_list_ir) = process_data_ir(reference_library_ir,
                                 calibration_set_ir,
                                 test_set_ir)



        
pred_slopp_df = corr_matching(np.array(slopp_df.iloc[:,:-4]), 
                              slopp_df['polymer'], 
                              np.array(test_set_raman.iloc[:,3:]), r_min=0)

pred_flopp_df = corr_matching(np.array(flopp_df.iloc[:,:-4]), 
                              flopp_df['polymer'], 
                              np.array(test_set_ir.iloc[:,3:]), r_min=0)


In [32]:
# Calculate HQI for calibration set
smx_cal = smx_nn(X_train_ir, X_cal_ir, label_train_ir)

# Calculate HQI for test set
smx_test = smx_nn(X_train_ir, X_test_ir, label_train_ir)

# Set confidence level 1 - alpha
n = smx_cal.shape[0]
alpha = 0.1
#for alpha in [alpha]:
prediction_sets, qhat = conform_pred(smx_cal, smx_test, label_cal_ir, n, alpha, cond=0)

prediction_sets_n = [list(label_table_ir.loc[ps,'polymer']) for ps in prediction_sets]
prediction_sets_size = np.array([len(ps) for ps in prediction_sets_n])

zero_label = np.where(prediction_sets_size == 0)[0]
single_label = np.where(prediction_sets_size == 1)[0]
mult_label = np.where(prediction_sets_size > 1)[0]

print('Sets with zero labels: ' + str(zero_label.shape[0]))
print('Sets with one label: ' + str(single_label.shape[0]))
print('Sets with >1 labels: ' + str(mult_label.shape[0]))
print('Mean set size:', prediction_sets.sum(1).mean())
print('Std set size:', prediction_sets.sum(1).std())

Sets with zero labels: 0
Sets with one label: 1600
Sets with >1 labels: 4517
Mean set size: 3.384829164623181
Std set size: 3.1960646920121527


In [507]:
output_df

,id,prection_set_ir,prection_set_raman,prection_set_mvcp,polymer_flopp,hqi_flopp,id_top_match_flopp,polymer_slopp,hqi_slopp,id_top_match_slopp
0,FHS_passive_August2024_B4_20240910_1,[Rubber],"[Acrylonitrile Butadiene Styrene, Polyamide, P...",[Rubber],Rubber,0.910802,R 16. Black Rubber Paving.CSV,Polymethyl Methacrylate,0.416396,Polymethyl Methacrylate 2. Clear Glasses Lens.txt
1,FHS_passive_August2024_B4_20240910_2,[Rubber],"[Acrylonitrile Butadiene Styrene, Cellulose Ac...",[Rubber],Rubber,0.914678,R 16. Black Rubber Paving.CSV,Polymethyl Methacrylate,0.409179,Polymethyl Methacrylate 2. Clear Glasses Lens.txt
2,FHS_passive_August2024_B4_20240910_3,[Rubber],"[Acrylonitrile Butadiene Styrene, Cellulose Ac...","[Polystyrene, Rubber]",Rubber,0.881395,R 16. Black Rubber Paving.CSV,Polymethyl Methacrylate,0.410750,Polymethyl Methacrylate 2. Clear Glasses Lens.txt
3,FHS_passive_August2024_B4_20240910_4,[Rubber],"[Acrylonitrile Butadiene Styrene, Polyamide, P...",[Rubber],Rubber,0.901583,R 16. Black Rubber Paving.CSV,Polymethyl Methacrylate,0.411261,Polymethyl Methacrylate 2. Clear Glasses Lens.txt
4,FHS_passive_August2024_B4_20240910_5,[Rubber],"[Acrylonitrile Butadiene Styrene, Cellulose Ac...","[Polystyrene, Rubber]",Rubber,0.900089,R 16. Black Rubber Paving.CSV,Polymethyl Methacrylate,0.409403,Polymethyl Methacrylate 2. Clear Glasses Lens.txt
...,...,...,...,...,...,...,...,...,...,...
4068,IAO_passive_June2024_G7_20240801_39,[Rubber],"[Acrylonitrile Butadiene Styrene, Cellulose Ac...",[Rubber],Rubber,0.944616,R 16. Black Rubber Paving.CSV,Polymethyl Methacrylate,0.407439,Polymethyl Methacrylate 2. Clear Glasses Lens.txt
4069,IAO_passive_June2024_G7_20240801_40,[Polystyrene],"[Acrylonitrile Butadiene Styrene, Cellulose Ac...","[Polypropylene, Polystyrene]",Polystyrene,0.697796,PS 1. Brown Coffee Cup Lid Fragment.CSV,Polymethyl Methacrylate,0.411182,Polymethyl Methacrylate 2. Clear Glasses Lens.txt
4070,IAO_passive_June2024_G7_20240801_41,[Rubber],"[Acrylonitrile Butadiene Styrene, Cellulose Ac...",[Rubber],Rubber,0.937212,R 16. Black Rubber Paving.CSV,Polymethyl Methacrylate,0.409412,Polymethyl Methacrylate 2. Clear Glasses Lens.txt
4071,IAO_passive_June2024_G7_20240801_42,[Rubber],"[Acrylonitrile Butadiene Styrene, Cellulose Ac...",[Rubber],Rubber,0.924160,R 16. Black Rubber Paving.CSV,Polymethyl Methacrylate,0.408313,Polymethyl Methacrylate 2. Clear Glasses Lens.txt


In [225]:
label_table_summary = label_table_raman.copy()
label_table_summary['res'] = 0
for i in range(label_table_raman.shape[0]):
    p = label_table_raman.iloc[i, 1]
    label_table_summary.iloc[i,2] = np.sum([p in x for x in prediction_sets_n_mvcp if len(x)==1])

label_table_summary.query('res>0')

,label,polymer,res
1,1,Acrylonitrile Butadiene Styrene,1
3,3,Polyamide,27
5,5,Polyethylene,20
6,6,Polyethylene Terephthalate,1
8,8,Polymethyl Methacrylate,3
9,9,Polypropylene,144
10,10,Polystyrene,152
11,11,Polyurethane,3
12,12,Polyvinyl Chloride,8
13,13,Rubber,1290
